# 09-2. Post-Training Quantization

01 에서 만든 **merged fp16 모델** 에 PTQ 를 적용하고 양자화 전후를 비교

| 구성 | 방식 | 분류 |
|---|---|---|
| fp16 | 양자화 없음 (baseline) | - |
| bnb-int8 | LLM.int8() weight-only | PTQ (on-the-fly) |
| bnb-nf4 | NF4 4bit weight-only | PTQ (on-the-fly) |
| GPTQ-int4 | 캘리브레이션 기반 4bit | PTQ (calibration) |

**비교 지표**: 디스크 용량 · 로딩 VRAM · 생성 속도(tok/s) · **Perplexity**

> bnb 는 로딩 시점에 즉석 양자화라 캘리브레이션이 없고, GPTQ 는 실제 데이터를 흘려보내
> **레이어별 오차를 보정**하며 양자화함 — 같은 4bit 여도 GPTQ 쪽 품질이 보통 더 좋음

**런타임: GPU**

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets huggingface_hub

In [ ]:
import os, json, time, gc, torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

HF_USER     = "본인_HF_아이디"          # TODO (01 과 동일)
MERGED_REPO = f"{HF_USER}/qwen2.5-1.5b-koalpaca-merged"
SAVE_DIR    = "/content/drive/MyDrive/ktb/09_llm_opt"

from google.colab import drive, userdata
drive.mount('/content/drive')
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

tokenizer = AutoTokenizer.from_pretrained(MERGED_REPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

with open(f"{SAVE_DIR}/eval_texts.json") as f:
    eval_texts = json.load(f)
len(eval_texts)

## 1. 평가 함수

- **Perplexity**: 모델이 정답 토큰에 얼마나 높은 확률을 주는지 → 낮을수록 좋음. 양자화 품질 열화의 주 지표
- **tok/s**: 동일 프롬프트로 128 토큰 생성 시간
- **VRAM**: 모델 로딩 직후 할당량

In [ ]:
PROMPT = tokenizer.apply_chat_template(
    [{"role": "user", "content": "머신러닝과 딥러닝의 차이를 설명해줘"}],
    tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def perplexity(model, texts, max_len=512):
    nll, ntok = 0.0, 0
    for t in texts:
        ids = tokenizer(t, return_tensors="pt", truncation=True,
                        max_length=max_len).input_ids.to(model.device)
        if ids.size(1) < 2:
            continue
        out = model(ids, labels=ids)
        n = ids.size(1) - 1
        nll += out.loss.item() * n     # loss 는 토큰 평균이므로 토큰 수를 곱해 누적
        ntok += n
    return float(torch.exp(torch.tensor(nll / ntok)))

@torch.no_grad()
def gen_speed(model, new_tokens=128):
    ids = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    model.generate(**ids, max_new_tokens=8)          # warm-up
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**ids, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    dt = time.time() - t0
    n = out.shape[1] - ids.input_ids.shape[1]
    return round(n / dt, 2), tokenizer.decode(out[0][ids.input_ids.shape[1]:],
                                              skip_special_tokens=True)

def dir_size_gb(path):
    total = sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(path) for f in fs)
    return round(total / 1024**3, 2)

RESULTS, SAMPLES = [], {}

def evaluate(tag, model, save_path=None):
    torch.cuda.synchronize()
    vram = round(torch.cuda.memory_allocated() / 1024**3, 2)
    tps, sample = gen_speed(model)
    ppl = perplexity(model, eval_texts[:50])
    size = dir_size_gb(save_path) if save_path else None
    RESULTS.append({"config": tag, "disk_GB": size, "vram_GB": vram,
                    "tok_per_sec": tps, "perplexity": round(ppl, 3)})
    SAMPLES[tag] = sample
    print(RESULTS[-1])

## 2. baseline — fp16

In [ ]:
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(MERGED_REPO, dtype=torch.float16,
                                             device_map={"": 0})
model.save_pretrained("/content/q_fp16"); tokenizer.save_pretrained("/content/q_fp16")
evaluate("fp16", model, "/content/q_fp16")

del model; torch.cuda.empty_cache(); gc.collect()

## 3. bitsandbytes INT8 / NF4

In [ ]:
def run_bnb(tag, cfg, path):
    global model
    torch.cuda.empty_cache(); gc.collect()
    model = AutoModelForCausalLM.from_pretrained(MERGED_REPO, quantization_config=cfg,
                                                 device_map={"": 0})
    model.save_pretrained(path); tokenizer.save_pretrained(path)
    evaluate(tag, model, path)
    del model; torch.cuda.empty_cache(); gc.collect()

run_bnb("bnb-int8", BitsAndBytesConfig(load_in_8bit=True), "/content/q_int8")

In [ ]:
run_bnb("bnb-nf4", BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
), "/content/q_nf4")

## 4. GPTQ INT4 (캘리브레이션 기반)

실제 데이터 128 샘플을 흘려보내며 레이어별 양자화 오차를 보정
설치·빌드가 환경을 타는 편이라 실패해도 위 결과만으로 과제 비교는 성립함

In [ ]:
!pip install -q gptqmodel --no-build-isolation

In [ ]:
from gptqmodel import GPTQModel, QuantizeConfig

calib = [t[:2048] for t in eval_texts[:128]]      # 캘리브레이션 데이터

qcfg = QuantizeConfig(bits=4, group_size=128)     # group_size: scale 공유 단위
gptq = GPTQModel.load(MERGED_REPO, qcfg)
gptq.quantize(calib, batch_size=1)
gptq.save("/content/q_gptq")
tokenizer.save_pretrained("/content/q_gptq")

del gptq; torch.cuda.empty_cache(); gc.collect()

In [ ]:
model = AutoModelForCausalLM.from_pretrained("/content/q_gptq", device_map={"": 0})
evaluate("gptq-int4", model, "/content/q_gptq")
del model; torch.cuda.empty_cache(); gc.collect()

## 5. 결과 비교

In [ ]:
df = pd.DataFrame(RESULTS).set_index("config")
df["size_ratio"] = (df["disk_GB"] / df.loc["fp16", "disk_GB"]).round(2)
df["ppl_증가율_%"] = ((df["perplexity"] / df.loc["fp16", "perplexity"] - 1) * 100).round(2)
df.to_csv(f"{SAVE_DIR}/metrics_ptq.csv")
df

In [ ]:
for k, v in SAMPLES.items():
    print("=" * 20, k, "=" * 20)
    print(v[:400], "\n")

**해석 포인트**
- 용량은 이론상 int8 = 1/2, int4 = 1/4 지만 **임베딩·LayerNorm 등은 양자화하지 않아** 정확히 떨어지진 않음
- bnb-int8 은 **오히려 느릴 수 있음** — outlier 를 fp16 으로 따로 처리하는 혼합 연산 오버헤드 때문
- 같은 4bit 라도 GPTQ 의 perplexity 증가폭이 bnb-nf4 보다 작으면 **캘리브레이션의 효과**를 입증한 것
- 수치와 별개로 **실제 생성 문장이 무너졌는지** 반드시 눈으로 확인 (PPL 만으로는 안 보임)